## Capstone Project: Student Performance Analysis

## Objective
You are a data analyst at a university. Using the full student dataset, produce a complete analysis report using **Pandas only** - from raw messy data to clean insights.

## What you will do
| Step | Task | Key Pandas concept |
|------|------|-------------------|
| 1 | Load & inspect data | `read_csv`, `info()`, `describe()` |
| 2 | Clean missing values | `fillna`, `dropna`, `isna()` |
| 3 | Engineer new features | `assign`, `apply`, `pd.cut` |
| 4 | Statistical summary | `groupby`, `agg`, `describe` |
| 5 | City-wise comparison | `groupby`, `pivot_table` |
| 6 | Top & bottom analysis | `nlargest`, `nsmallest`, `sort_values` |
| 7 | Study hours insight | `corr()`, `groupby`, `pd.cut` |
| 8 | Subject analysis (long format) | `melt`, `groupby` |
| 9 | Gender comparison | Boolean mask groupby |
| 10 | Final report | Method chaining, `to_csv` |


In [1]:
import pandas as pd
import numpy as np

#### STEP 1: LOAD & INSPECT 

In [3]:
df = pd.read_csv('students.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Missing values:\n{df.isna().sum()[df.isna().sum()>0]}')

Shape: (50, 10)
Columns: ['student_id', 'name', 'gender', 'age', 'study_hours', 'maths', 'science', 'english', 'compSci', 'attendance']
Missing values:
maths      4
science    4
english    4
compSci    4
dtype: int64


#### STEP 2: CLEAN

In [5]:
score_cols = ['maths','science','english','compSci']

df_clean = (
    df
    .assign(**{col: df[col].fillna(df[col].median()) for col in score_cols})
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f'After cleaning: {df_clean.shape}  |  NaN: {df_clean.isna().sum().sum()}')

After cleaning: (50, 10)  |  NaN: 0


#### STEP 3: FEATURE ENGINEERING

In [6]:
def assign_grade(df_):
    conds = [df_['average']>=85, df_['average']>=70, df_['average']>=55, df_['average']>=40]
    return np.select(conds, ['A','B','C','D'], default='F')

df_clean = (
    df_clean
    .assign(
        total   = lambda x: x[score_cols].sum(axis=1).round(1),
        average = lambda x: x[score_cols].mean(axis=1).round(2),
        passed  = lambda x: x[score_cols].mean(axis=1) >= 40,
        grade   = assign_grade,
        study_group = lambda x: pd.cut(x['study_hours'],
                                       bins=[0,6,10,15],
                                       labels=['Low','Medium','High']),
        strongest_sub = lambda x: x[score_cols].idxmax(axis=1)
    )
)
print('New columns added:', ['total','average','passed','grade','study_group','strongest_sub'])
print(df_clean[['name','average','grade','study_group','strongest_sub']].head())

New columns added: ['total', 'average', 'passed', 'grade', 'study_group', 'strongest_sub']
     name  average grade study_group strongest_sub
0   Aarav    69.55     C         Low       compSci
1   Aanya    76.28     B      Medium       science
2   Aditi    63.95     C      Medium         maths
3   Arjun    69.92     C      Medium       english
4  Bhavna    64.55     C        High       science


#### STEP 4: STATISTICAL SUMMARY

In [8]:
print(df_clean[score_cols + ['average']].describe().round(2))

        maths  science  english  compSci  average
count   50.00    50.00    50.00    50.00    50.00
mean    64.78    70.04    72.02    70.52    69.34
std     13.52    12.00     9.99    13.87     6.92
min     36.20    47.10    49.60    41.60    52.08
25%     57.42    58.45    64.70    61.08    65.26
50%     64.80    71.50    73.40    70.55    69.15
75%     70.80    77.68    78.58    79.20    74.16
max    100.00   100.00    99.80   100.00    87.88


In [9]:
df_clean = pd.read_csv('students_enriched.csv')

In [10]:
score_cols = ['maths','science','english','compSci']

for col in score_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

if 'average' not in df_clean.columns:
    df_clean['average'] = df_clean[score_cols].mean(axis=1).round(2)

if 'grade' not in df_clean.columns:
    conds = [
        df_clean['average'] >= 85,
        df_clean['average'] >= 70,
        df_clean['average'] >= 55,
        df_clean['average'] >= 40
    ]
    df_clean['grade'] = np.select(conds, ['A','B','C','D'], default='F')

if 'passed' not in df_clean.columns:
    df_clean['passed'] = df_clean['average'] >= 40

#### STEP 5: GENDER COMPARISON

In [11]:
gender_stats = df_clean.groupby('gender').agg(
    students   = ('name','count'),
    avg_score  = ('average','mean'),
    pass_rate  = ('passed','mean'),
    top_score  = ('average','max')
).round(3)

gender_stats['pass_pct'] = (gender_stats['pass_rate']*100).round(1).astype(str)+'%'
print(gender_stats.drop(columns='pass_rate').sort_values('avg_score',ascending=False))


        students  avg_score  top_score pass_pct
gender                                         
Male          27     70.703      87.88   100.0%
Female        23     67.600      84.57   100.0%


#### STEP 6: TOP & B0TTOM

In [12]:
print('Top 5 students:')
print(df_clean.nlargest(5,'average')[['name','average','grade']].to_string(index=False))

print()

print('Bottom 5 students:')
print(df_clean.nsmallest(5,'average')[['name','average','grade']].to_string(index=False))


Top 5 students:
  name  average grade
 Naina    87.88     A
 Tanvi    84.57     B
Shreya    84.20     B
 Rahul    76.90     B
   Jay    76.82     B

Bottom 5 students:
     name  average grade
Siddharth    52.08     D
    Pooja    56.85     C
    Meera    59.65     C
    Sneha    59.78     C
    Kavya    60.92     C


#### STEP 7: STUDY HOURS INSIGHT

In [14]:
print(f'Correlation (study_hours vs average): {df_clean["study_hours"].corr(df_clean["average"]):.4f}')

study_grp_stats = df_clean.groupby(
    pd.cut(df_clean['study_hours'],
           bins=[0,6,10,15],
           labels=['Low(0-6)','Mid(7-10)','High(11+)'])
)['average'].agg(['mean','count']).round(2)

print('\nAverage by study group:')
print(study_grp_stats)


Correlation (study_hours vs average): 0.0667

Average by study group:
              mean  count
study_hours              
Low(0-6)     70.06     13
Mid(7-10)    67.30     18
High(11+)    70.62     19


C:\Users\Purvi jain\AppData\Local\Temp\ipykernel_33604\1175968815.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  study_grp_stats = df_clean.groupby(


#### STEP 8: SUBJECT ANALYSIS

In [15]:
df_long = df_clean.melt(
    id_vars=['name','gender'],
    value_vars=score_cols,
    var_name='subject',
    value_name='score'
)

sub_stats = df_long.groupby('subject')['score'].agg(['mean','std','min','max']).round(2)
sub_stats = sub_stats.sort_values('mean',ascending=False)

print(sub_stats)

print()
print('Most common strongest subject:')

strong = df_clean[score_cols].idxmax(axis=1).value_counts()
print(strong)

          mean    std   min    max
subject                           
english  71.90   9.98  49.6   99.8
compSci  70.52  13.87  41.6  100.0
science  69.91  11.99  47.1  100.0
maths    64.78  13.52  36.2  100.0

Most common strongest subject:
compSci    14
science    14
english    12
maths      10
Name: count, dtype: int64


#### STEP 9: GENDER PERFORMANCE

In [16]:
gender_stats = df_clean.groupby('gender').agg(
    count     = ('name','count'),
    avg_score = ('average','mean'),
    pass_rate = ('passed','mean'),
    avg_maths = ('maths','mean')
).round(3)

gender_stats['pass_pct'] = (gender_stats['pass_rate']*100).round(1).astype(str)+'%'
print(gender_stats)

        count  avg_score  pass_rate  avg_maths pass_pct
gender                                                 
Female     23     67.600        1.0     65.743   100.0%
Male       27     70.703        1.0     63.956   100.0%


#### FINAL REPORT

In [17]:
weakest_sub = df_long.groupby('subject')['score'].mean().idxmin()
best_sub    = df_long.groupby('subject')['score'].mean().idxmax()
top_student = df_clean.loc[df_clean['average'].idxmax(),'name']

grade_dist = df_clean['grade'].value_counts().to_dict()

print(f'  Total students    : {len(df_clean)}')
print(f'  Overall avg score : {df_clean["average"].mean():.2f}')
print(f'  Pass rate         : {df_clean["passed"].mean()*100:.1f}%')
print(f'  Distinction (A)   : {grade_dist.get("A",0)} students')
print(f'  Fail (F)          : {grade_dist.get("F",0)} students')
print(f'  Best subject      : {best_sub}')
print(f'  Weakest subject   : {weakest_sub}')
print(f'  Top student       : {top_student}')
print(f'  Study-score corr  : {df_clean["study_hours"].corr(df_clean["average"]):.4f}')

df_clean.to_csv('students_final.csv', index=False)

  Total students    : 50
  Overall avg score : 69.28
  Pass rate         : 100.0%
  Distinction (A)   : 1 students
  Fail (F)          : 0 students
  Best subject      : english
  Weakest subject   : maths
  Top student       : Naina
  Study-score corr  : 0.0667
